# P4 — Evasión de Colisiones (Problema Abierto)
## MyCobot 280 · 6 DOF

**Estrategia elegida:** combinación de tres mecanismos preventivos
1. Límites conservadores por joint
2. Verificación de altura mínima mediante FK completa (DH 6-DOF)
3. Waypoint seguro de clearance (con verificación propia)

---

## 0. Imports y conexión

In [ ]:
import time
import numpy as np
from math import radians, cos, sin
from pymycobot.mycobot import MyCobot

mc = MyCobot('/dev/ttyUSB0', 1000000)
time.sleep(1)
mc.power_on()
time.sleep(1)
print('Conexión:', mc.is_controller_connected())

---
## 1. Cinemática Directa completa (DH 6-DOF)

Se usa la tabla DH de referencia con `theta_offset` para calcular la posición
real del efector. Esto reemplaza la FK simplificada (planar 3-DOF) del script anterior,
que usaba valores desactualizados y no consideraba los offsets de la pose de referencia.

In [ ]:
# Tabla DH: (a_mm, d_mm, alpha_deg, theta_offset_deg)
# Fuente: pose de referencia del JetCobot (ctrl_joints)
DH_TABLE = [
    (   0,  134.75,  90.0,   0.0),   # J1 Base
    (-110,    0.00,   0.0, -90.0),   # J2 Hombro
    ( -96,    0.00,   0.0,   0.0),   # J3 Codo
    (   0,   63.40,  90.0, -90.0),   # J4 Muñeca 1
    (   0,   75.05, -90.0,  90.0),   # J5 Muñeca 2
    (   0,   50.00,   0.0,   0.0),   # J6 Gripper
]


def dh_matrix(theta_deg, d, a, alpha_deg):
    """Matriz de transformación homogénea DH estándar 4x4."""
    t = radians(theta_deg)
    al = radians(alpha_deg)
    ct, st = cos(t), sin(t)
    ca, sa = cos(al), sin(al)
    return np.array([
        [ct, -st*ca,  st*sa, a*ct],
        [st,  ct*ca, -ct*sa, a*st],
        [ 0,     sa,     ca,    d],
        [ 0,      0,      0,    1],
    ], dtype=float)


def forward_kinematics(angles_deg):
    """
    Calcula T0_6 aplicando los theta_offset de la pose de referencia.
    angles_deg: ángulos de hardware [q1..q6] leídos con mc.get_angles()
    """
    T = np.eye(4)
    for i, (a, d, alpha, offset) in enumerate(DH_TABLE):
        T = T @ dh_matrix(angles_deg[i] + offset, d, a, alpha)
    return T


def get_z(angles_deg):
    """Devuelve la coordenada Z del efector en mm."""
    T = forward_kinematics(angles_deg)
    return T[2, 3]


# Verificación en home [0,0,0,0,0,0]
T_home = forward_kinematics([0]*6)
x, y, z = T_home[0,3], T_home[1,3], T_home[2,3]
print('FK completa (DH 6-DOF) — pose HOME [0,0,0,0,0,0]:')
print(f'  x = {x:.2f} mm')
print(f'  y = {y:.2f} mm')
print(f'  z = {z:.2f} mm')
print(f'\nMatriz T0_6:\n{np.round(T_home, 3)}')

---
## 2. Escenarios de colisión identificados

Se identificaron tres tipos de colisión en el entorno de laboratorio real
y para cada uno se justifica el mecanismo elegido.

In [ ]:
COLLISION_SCENARIOS = {
    'colision_mesa': {
        'descripcion': 'El efector o los eslabones golpean la superficie de la mesa.',
        'causa'      : 'J2/J3 llevan el brazo por debajo de Z=60 mm.',
        'mecanismo'  : 'Mecanismo 2 — verificación de Z mínima por FK.',
        'por_que'    : 'FK calcula la Z real del efector antes de ejecutar '
                       'el movimiento. Es más preciso que solo revisar ángulos.'
    },
    'auto_colision': {
        'descripcion': 'El codo golpea el hombro o el gripper toca el cuerpo del robot.',
        'causa'      : 'J2 y J3 en valores extremos simultáneos.',
        'mecanismo'  : 'Mecanismo 1 — límites conservadores por joint.',
        'por_que'    : 'Reducir los rangos físicos en ~15% elimina las '
                       'configuraciones peligrosas sin necesidad de sensores adicionales.'
    },
    'colision_objetos': {
        'descripcion': 'El brazo choca con contenedores, cámara o cables de la mesa.',
        'causa'      : 'Trayectoria directa entre pick y place cruza zona ocupada.',
        'mecanismo'  : 'Mecanismo 3 — waypoint seguro de clearance.',
        'por_que'    : 'Forzar un punto intermedio elevado garantiza que el brazo '
                       'pase siempre por encima de cualquier obstáculo fijo.'
    },
}

print('ESCENARIOS DE COLISIÓN — MyCobot 280')
print('=' * 65)
for nombre, info in COLLISION_SCENARIOS.items():
    print(f'\n[{nombre}]')
    for k, v in info.items():
        print(f'  {k:<12}: {v}')
print('=' * 65)
print('\nJustificación de la combinación elegida:')
print('  Los 3 mecanismos son complementarios y sin superposición:')
print('  - Mecanismo 1 actúa en espacio de joints (rápido, sin cómputo)')
print('  - Mecanismo 2 actúa en espacio cartesiano (preciso, via FK)')
print('  - Mecanismo 3 actúa en la trayectoria (cobertura de objetos externos)')
print('  Juntos cubren colisiones con mesa, auto-colisiones y obstáculos fijos.')

---
## 3. Mecanismo 1 — Límites conservadores por joint

In [ ]:
# Límites físicos del robot vs límites conservadores usados
JOINT_LIMITS_FISICOS = [
    (-168, 168), (-135,  90), (-150, 150),
    (-145, 145), (-165, 165), (-180, 180),
]

CONSERVATIVE_JOINT_LIMITS = [
    (-160, 160),   # J1: -168/+168 → reducido ~5%
    (-110,  70),   # J2: -135/+90  → más restrictivo (zona de auto-colisión)
    (-120, 120),   # J3: -150/+150 → reducido ~20%
    (-130, 130),   # J4: -145/+145 → reducido ~10%
    (-150, 150),   # J5: -165/+165 → reducido ~9%
    (-175, 175),   # J6: -180/+180 → reducido ~3%
]


def check_joint_limits(angles):
    """Verifica que todos los ángulos estén dentro de los límites conservadores."""
    for i, angle in enumerate(angles):
        lo, hi = CONSERVATIVE_JOINT_LIMITS[i]
        if not (lo <= angle <= hi):
            return False, f'J{i+1}={angle:.1f}° fuera de [{lo}, {hi}]'
    return True, 'Todos los joints dentro de límites'


print('Límites por joint (físico → conservador):')
print(f'  {"Joint":<8} {"Físico":>18}  {"Conservador":>18}  {"Reducción"}')
print('-' * 65)
for i, (fis, con) in enumerate(zip(JOINT_LIMITS_FISICOS, CONSERVATIVE_JOINT_LIMITS)):
    red_lo = fis[0] - con[0]
    red_hi = con[1] - fis[1]
    print(f'  J{i+1:<7} [{fis[0]:>5},{fis[1]:>4}]     [{con[0]:>5},{con[1]:>4}]     '
          f'+{red_lo}° / -{red_hi}°')
print('-' * 65)

---
## 4. Mecanismo 2 — Verificación de altura mínima por FK completa

In [ ]:
Z_MIN_SAFE = 60.0   # mm — altura mínima segura sobre la mesa


def check_min_height(angles):
    """
    Calcula Z del efector con FK 6-DOF completa y verifica Z >= Z_MIN_SAFE.
    Retorna (valido, z_calculado, mensaje).
    """
    z = get_z(angles)
    if z < Z_MIN_SAFE:
        return False, z, f'Z={z:.1f} mm < Z_min={Z_MIN_SAFE} mm — PELIGROSO'
    return True, z, f'Z={z:.1f} mm >= Z_min={Z_MIN_SAFE} mm — seguro'


# Comparación: FK simplificada anterior vs FK completa actual
test_angles = [0, -100, 50, 0, 0, 0]

# FK simplificada (como estaba antes)
L1, L2_s, L3_s = 131.0, 110.0, 96.0
t2, t3 = radians(test_angles[1]), radians(test_angles[2])
z_simple = L1 + L2_s*sin(t2) + L3_s*sin(t2 + t3)

# FK completa (DH 6-DOF con offsets)
z_completa = get_z(test_angles)

print(f'Comparación FK para {test_angles}:')
print(f'  FK simplificada (anterior) : Z = {z_simple:.2f} mm')
print(f'  FK completa DH 6-DOF       : Z = {z_completa:.2f} mm')
print(f'  Diferencia                 : {abs(z_completa - z_simple):.2f} mm')
print()

ok, z_val, msg = check_min_height(test_angles)
print(f'Resultado check_min_height: {msg}')

---
## 5. Mecanismo 3 — Waypoint seguro de clearance (con verificación propia)

In [ ]:
# Waypoint intermedio elevado — el robot pasa por aquí entre pick y place
SAFE_WAYPOINT = [0.0, 0.0, -90.0, 90.0, 0.0, -45.0]

# Verificar que el propio waypoint es seguro (corrección del script anterior)
ok_l, msg_l = check_joint_limits(SAFE_WAYPOINT)
ok_z, z_wp,  msg_z = check_min_height(SAFE_WAYPOINT)

print('Verificación del SAFE_WAYPOINT:')
print(f'  Ángulos : {SAFE_WAYPOINT}')
print(f'  Límites : {"OK" if ok_l else "FALLO"} — {msg_l}')
print(f'  Altura  : {"OK" if ok_z else "FALLO"} — {msg_z}')

if ok_l and ok_z:
    print('  → Waypoint validado: es seguro usarlo como clearance.')
else:
    print('  → ADVERTENCIA: el waypoint no es seguro, ajustar SAFE_WAYPOINT.')

---
## 6. CollisionChecker — integración de los 3 mecanismos

In [ ]:
class CollisionChecker:
    """
    Sistema de prevención de colisiones para el MyCobot 280.

    Integra 3 mecanismos complementarios:
      1. check_joint_limits()  — límites conservadores en espacio de joints
      2. check_min_height()    — Z mínima del efector via FK 6-DOF completa
      3. safe_move()           — waypoint de clearance (verificado) entre poses
    """

    def __init__(self, mc=None):
        self.mc = mc

    def check_joint_limits(self, angles):
        return check_joint_limits(angles)

    def check_min_height(self, angles):
        return check_min_height(angles)

    def safe_move(self, target_angles, speed=30):
        """
        Ejecuta un movimiento seguro en 4 pasos:
          1. Valida límites de joints del destino
          2. Valida Z mínima del destino
          3. Valida que el SAFE_WAYPOINT sea seguro
          4. Ejecuta: waypoint → destino
        """
        # Paso 1: límites del destino
        ok_l, msg_l = self.check_joint_limits(target_angles)
        if not ok_l:
            print(f'  [BLOQUEADO - límites] {msg_l}')
            return False

        # Paso 2: altura mínima del destino
        ok_z, z_val, msg_z = self.check_min_height(target_angles)
        if not ok_z:
            print(f'  [BLOQUEADO - altura]  {msg_z}')
            return False

        # Paso 3: verificar el propio waypoint
        ok_wl, _ = self.check_joint_limits(SAFE_WAYPOINT)
        ok_wz, z_wp, _ = self.check_min_height(SAFE_WAYPOINT)
        if not (ok_wl and ok_wz):
            print(f'  [ERROR] SAFE_WAYPOINT no es seguro (Z={z_wp:.1f} mm)')
            return False

        print(f'  [OK] destino Z={z_val:.1f} mm — ejecutando')

        # Paso 4: mover con hardware
        if self.mc is not None:
            self.mc.send_angles(SAFE_WAYPOINT, speed)
            time.sleep(3)
            self.mc.send_angles(target_angles, speed)
            time.sleep(3)

        return True


checker = CollisionChecker(mc=mc)
print('CollisionChecker inicializado con FK 6-DOF completa.')

---
## 7. Pruebas con el robot real

In [ ]:
test_cases = [
    ([  0,   0,   0,   0,   0, -45], 'HOME — debe pasar'),
    ([ 30,   0,   0, -45,   0,   0], 'Pose normal — debe pasar'),
    ([  0,-120,   0,   0,   0,   0], 'J2=-120 fuera de límite — bloqueado'),
    ([  0,-100,  50,   0,   0,   0], 'Z muy baja — bloqueado por altura'),
    ([  0, -80,  80,  90,   0,   0], 'Pose baja — verificar Z'),
]

print('=' * 65)
print('PRUEBAS DE EVASIÓN DE COLISIONES — Robot real')
print('=' * 65)

resultados = []
for angles, desc in test_cases:
    print(f'\n[{desc}]')
    print(f'  Ángulos: {angles}')

    ok_l, msg_l = checker.check_joint_limits(angles)
    ok_z, z_val, msg_z = checker.check_min_height(angles)

    print(f'  Límites: {"OK" if ok_l else "FALLO"} — {msg_l}')
    print(f'  Altura : {"OK" if ok_z else "FALLO"} — {msg_z}')

    if ok_l and ok_z:
        resultado = checker.safe_move(angles, speed=30)
    else:
        resultado = False

    resultados.append((desc, ok_l, ok_z, resultado))

print('\n' + '=' * 65)
print('RESUMEN')
print('=' * 65)
print(f'  {"Caso":<35} {"Límites":>8} {"Altura":>8} {"Ejecutado":>10}')
print('-' * 65)
for desc, ok_l, ok_z, res in resultados:
    print(f'  {desc:<35} {"OK" if ok_l else "NO":>8} '
          f'{"OK" if ok_z else "NO":>8} {"SI" if res else "NO":>10}')
print('=' * 65)

---
## 8. Verificación cruzada: FK vs coordenadas reales del robot

In [ ]:
# Lee la posición actual del robot y compara FK calculada vs robot
angles_act = mc.get_angles()
time.sleep(0.3)
coords_robot = mc.get_coords()

T_act = forward_kinematics(angles_act)
x_fk, y_fk, z_fk = T_act[0,3], T_act[1,3], T_act[2,3]

print('Verificación cruzada FK vs robot:')
print(f'  Ángulos actuales : {[round(a,2) for a in angles_act]}')
print()
print(f'  {"Eje":<5} {"FK calculada":>14} {"Robot real":>14} {"Error":>10}')
print('-' * 50)
ejes = [('X', x_fk), ('Y', y_fk), ('Z', z_fk)]
for i, (eje, fk_v) in enumerate(ejes):
    rob_v = coords_robot[i] if coords_robot else float('nan')
    err   = abs(fk_v - rob_v) if coords_robot else float('nan')
    print(f'  {eje:<5} {fk_v:>14.2f} {rob_v:>14.2f} {err:>10.2f} mm')
print('-' * 50)
if coords_robot:
    err_total = np.linalg.norm([x_fk-coords_robot[0],
                                y_fk-coords_robot[1],
                                z_fk-coords_robot[2]])
    print(f'  Error euclidiano: {err_total:.2f} mm')
    veredicto = 'OK (< 10 mm)' if err_total < 10 else 'REVISAR tabla DH'
    print(f'  Veredicto FK    : {veredicto}')

---
## 9. Conclusiones

In [ ]:
print('=' * 65)
print('CONCLUSIONES — P4 Evasión de Colisiones')
print('=' * 65)
conclusiones = [
    ('Escenarios identificados',
     'colisión con mesa, auto-colisión J2/J3, colisión con objetos externos.'),
    ('Mecanismo 1 — límites conservadores',
     'Reduce rangos físicos ~15% eliminando configuraciones peligrosas '
     'sin cómputo adicional. Actúa en espacio de joints.'),
    ('Mecanismo 2 — verificación Z por FK',
     'Usa FK DH 6-DOF completa con theta_offsets de la pose de referencia. '
     'Más precisa que la FK simplificada anterior (diferencia hasta ~30 mm).'),
    ('Mecanismo 3 — waypoint verificado',
     'El propio SAFE_WAYPOINT es validado por los mecanismos 1 y 2 antes '
     'de usarlo, eliminando el supuesto de que el waypoint es siempre seguro.'),
    ('Limitaciones',
     'Sin sensores externos, FK no modela el volumen del brazo (solo TCP), '
     'no existe planificación dinámica de trayectorias.'),
    ('Resultado',
     'Movimientos peligrosos bloqueados correctamente. '
     'Sistema adecuado para entorno controlado de laboratorio.'),
]
for titulo, texto in conclusiones:
    print(f'\n  [{titulo}]')
    print(f'  {texto}')
print('\n' + '=' * 65)